# Cohort Shift Alignment ($\rho_b$ sweep)

Reproduces Table `tab:app-rhob` (Appendix, Sensitivity Analyses). $\rho_b$ is the alignment between the cohort shift direction $\mu_b$ and the confounding direction $\bar{w}_b$:
- $\rho_b=1.0$: fully aligned (main experiment baseline regime)
- $\rho_b=0.5$: partial alignment
- $\rho_b=0.0$: random shift

**Setup:** baseline regime, $N_{\text{eval}}=1000$, S1 and S2B configurations, 50 seeds × 7 models, shift magnitude held fixed. Entangled = mean of CCA and CLIP. Factorized = mean of JIVE and DSSL, with DSSL averaged over its four $\beta$ configurations.

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP


def fmt(v: float, nd: int = 3) -> str:
    """Format with round-half-up, matching the paper (exact ties such as 0.5975 occur)."""
    return str(Decimal(f'{v:.10f}').quantize(Decimal(1).scaleb(-nd), rounding=ROUND_HALF_UP))

DATA_DIR = Path('../../data')
BA_DIR  = DATA_DIR / 'batch_align_ablation/results'
LIN_DIR = DATA_DIR / 'main_experiment/results'

PURE = ('s1_', 's2b_')
GT   = {'s1_': 'scenario_1', 's2b_': 'scenario_2'}

def get_gt(cfg):
    for p, s in GT.items():
        if cfg.startswith(p): return s

def model_family(m):
    if m.startswith('cca'):  return 'CCA'
    if m.startswith('clip'): return 'CLIP'
    if m.startswith('jive'): return 'JIVE'
    return 'DSSL'

# Load ρ_b ∈ {0.0, 0.5}
records = []
for f in sorted(BA_DIR.glob('*.json')):
    rho = 0.0 if f.stem.startswith('ba00') else 0.5
    for r in json.load(open(f)):
        if r['n_eval'] == 1000 and any(r['alpha_config_id'].startswith(p) for p in PURE):
            r['rho_b'] = rho
            records.append(r)

# Load ρ_b=1.0 from main experiment baseline
for f in sorted(LIN_DIR.glob('baseline_baseline_*.json')):
    for r in json.load(open(f)):
        if r['n_eval'] == 1000 and any(r['alpha_config_id'].startswith(p) for p in PURE):
            r['rho_b'] = 1.0
            records.append(r)

df = pd.DataFrame(records)
df['ground_truth']  = df['alpha_config_id'].map(get_gt)
df['model_family']  = df['model'].map(model_family)

print(f'Total records: {len(df)}')
print(f'Per ρ_b: {dict(df["rho_b"].value_counts().sort_index())}')

In [ ]:
RHO_VALUES = [1.0, 0.5, 0.0]
FAMILY_GROUPS = {'Entangled': ['CCA', 'CLIP'], 'Factorized': ['JIVE', 'DSSL']}

rows = []
for group, members in FAMILY_GROUPS.items():
    for rho in RHO_VALUES:
        per_model = []
        for fam in members:
            sub = df[(df['rho_b'] == rho) & (df['model_family'] == fam)]
            s1 = sub[sub['ground_truth'] == 'scenario_1']['predicted_scenario']
            s2b = sub[sub['ground_truth'] == 'scenario_2']['predicted_scenario']
            per_model.append({
                'S1 acc': (s1 == 'scenario_1').mean(),
                'S2B acc': (s2b == 'scenario_2').mean(),
                'S2B to S1': (s2b == 'scenario_1').mean(),
                'S2B to indeterminate': (s2b == 'indeterminate').mean(),
            })
        rows.append({'model': group, 'rho_b': rho, **pd.DataFrame(per_model).mean().to_dict()})

table = pd.DataFrame(rows).set_index(['model', 'rho_b'])
print(table.to_string(formatters={
    'S1 acc': fmt, 'S2B acc': fmt,
    'S2B to S1': lambda v: fmt(100 * v, 0) + '%',
    'S2B to indeterminate': lambda v: fmt(100 * v, 0) + '%',
}))